In [1]:
import requests
from sentence_transformers import SentenceTransformer
from endee import Endee,Precision

In [2]:
resp = requests.get("http://localhost:8080")
print(resp.status_code)

200


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Vector databases enable semantic search.",
    "Docker helps deploy applications consistently.",
    "Endee is a high-performance vector database."
]

In [5]:
embeddings = model.encode(documents)

In [6]:
client = Endee()

In [7]:
client.create_index(name="docs",dimension = model.get_sentence_embedding_dimension() ,space_type = 'cosine',precision=Precision.INT8D)

'Index created successfully'

In [8]:
embeddings = embeddings.tolist()

In [9]:
index = client.get_index(name = 'docs')

In [10]:
payload = [
        {
            "id": f"doc{i}",
            "vector": embeddings[i],
            "metadata": {
                "text": documents[i],
                "category": "tech"
            }
        }
        for i in range(len(documents[:4]))
    ]

In [14]:
index.upsert(payload)

'Vectors inserted successfully'

In [15]:
results=index.query(vector = embeddings[4],top_k=5)

In [16]:
for item in results:
    print(f"ID: {item['id']}, Similarity: {item['similarity']}")

ID: doc2, Similarity: 0.5729258060455322
ID: doc1, Similarity: 0.3068729043006897
ID: doc0, Similarity: 0.20017372071743011
ID: doc3, Similarity: 0.1665257066488266
